# Column audit for `cleaned_metrics.csv`

This notebook creates a **read-only audit** of every input column. It describes data quality, group availability, and a conservative initial Machine Learning role. It does not clean, fill, transform, label, or model the data.

### 1. Import libraries and define paths

**Objective:** Load the small set of libraries used for tabular inspection and define reproducible repository-relative paths.  
**What this checks:** It verifies that the expected input file exists and prepares the output directory.  
**Why this matters for ML:** Explicit paths make an analysis reproducible and reduce the risk of silently auditing the wrong dataset.

In [1]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
INPUT_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_metrics.csv"
OUTPUT_PATH = PROJECT_ROOT / "reports" / "column_audit.csv"

if not INPUT_PATH.is_file():
    raise FileNotFoundError(f"Expected input dataset was not found: {INPUT_PATH}")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f"Input file found: {INPUT_PATH}")
print(f"Audit will be saved to: {OUTPUT_PATH}")

Input file found: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/data/processed/cleaned_metrics.csv
Audit will be saved to: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/column_audit.csv


### 2. Load the dataset without modifying it

**Objective:** Read the source CSV into memory exactly once and record a SHA-256 fingerprint of its bytes.  
**What this checks:** The fingerprint lets the final cell verify that the original file did not change during the audit.  
**Why this matters for ML:** Auditing must be observational; mutating source data would make later experiments difficult to reproduce.

In [2]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

input_hash_before = sha256_file(INPUT_PATH)
metrics_df = pd.read_csv(INPUT_PATH)

print(f"Loaded {len(metrics_df):,} rows and {metrics_df.shape[1]} columns.")
print(f"Input SHA-256 before audit: {input_hash_before}")

Loaded 31,059 rows and 39 columns.
Input SHA-256 before audit: 801976ddbe91a7415ce84038ff7df5b89c601e2201e56eaf5503154f93c2e577


### 3. Inspect the dataset structure

**Objective:** Display the shape, complete column list, and first five rows.  
**What this checks:** It provides a quick visual validation that parsing produced the expected table and headers.  
**Why this matters for ML:** Row/column counts and sample records often reveal delimiter, header, type, or schema problems before feature work begins.

In [3]:
print("Dataset shape:", metrics_df.shape)
print("\nColumn names:")
for position, column_name in enumerate(metrics_df.columns, start=1):
    print(f"{position:>2}. {column_name}")

print("\nFirst five rows:")
display(metrics_df.head())

Dataset shape: (31059, 39)

Column names:
 1. id
 2. run_id
 3. machine_id
 4. timestamp
 5. elapsed_seconds
 6. phase
 7. stress_cpu_target_pct
 8. stress_memory_target_mb
 9. cpu_pct
10. cpu_per_core_json
11. cpu_frequency_mhz
12. ram_pct
13. ram_used_mb
14. ram_available_mb
15. swap_pct
16. swap_used_mb
17. disk_usage_pct
18. disk_free_gb
19. disk_read_mb_s
20. disk_write_mb_s
21. disk_latency_ms
22. net_sent_mb_s
23. net_recv_mb_s
24. network_latency_ms
25. process_count
26. thread_count
27. context_switches_per_s
28. temperature_c
29. gpu_usage_pct
30. gpu_per_device_json
31. battery_pct
32. battery_plugged
33. sensor_errors_json
34. missed_deadline
35. legacy_id
36. sample_reliable
37. status
38. ended_at_utc
39. run_complete

First five rows:


,id,run_id,machine_id,timestamp,elapsed_seconds,phase,stress_cpu_target_pct,stress_memory_target_mb,cpu_pct,cpu_per_core_json,...,gpu_per_device_json,battery_pct,battery_plugged,sensor_errors_json,missed_deadline,legacy_id,sample_reliable,status,ended_at_utc,run_complete
0,1,6835f125-a038-4092-beff-5107ae998b39,0890dcc046c079acc4de4202,2026-07-25T14:45:59.803000Z,1.022,monitor,NaN,NaN,0.0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",...,[],49.0,0,"{""network_latency"": ""TimeoutError: timed out"",...",0,NaN,0,completed,2026-07-25T16:45:58.885Z,1
1,2,6835f125-a038-4092-beff-5107ae998b39,0890dcc046c079acc4de4202,2026-07-25T14:46:00.857000Z,2.076,monitor,NaN,NaN,9.8,"[19.2, 18.6, 15.4, 10.5, 8.5, 3.0, 2.0, 11.1, ...",...,[],49.0,0,"{""temperature"": ""macmon_not_installed (install...",0,NaN,0,completed,2026-07-25T16:45:58.885Z,1
2,3,6835f125-a038-4092-beff-5107ae998b39,0890dcc046c079acc4de4202,2026-07-25T14:46:02.829000Z,4.048,monitor,NaN,NaN,5.4,"[15.6, 12.6, 10.6, 8.5, 1.0, 0.0, 0.0, 3.0, 1....",...,[],49.0,0,"{""temperature"": ""macmon_not_installed (install...",0,NaN,0,completed,2026-07-25T16:45:58.885Z,1
3,4,6835f125-a038-4092-beff-5107ae998b39,0890dcc046c079acc4de4202,2026-07-25T14:46:04.845000Z,6.063,monitor,NaN,NaN,4.6,"[17.0, 11.5, 9.0, 6.0, 0.5, 0.0, 0.0, 1.5, 1.0...",...,[],49.0,0,"{""temperature"": ""macmon_not_installed (install...",0,NaN,0,completed,2026-07-25T16:45:58.885Z,1
4,5,6835f125-a038-4092-beff-5107ae998b39,0890dcc046c079acc4de4202,2026-07-25T14:46:06.853000Z,8.071,monitor,NaN,NaN,7.3,"[20.2, 16.6, 10.1, 14.6, 1.0, 0.5, 0.5, 5.0, 3...",...,[],49.0,0,"{""temperature"": ""macmon_not_installed (install...",0,NaN,0,completed,2026-07-25T16:45:58.885Z,1


### 4. Define reusable audit helpers and conservative role rules

**Objective:** Make the audit logic explicit before applying it.  
**What this checks:** Helpers safely format examples, summarize group coverage, and map known columns to one allowed role; unknown columns are deliberately assigned `review_needed`.  
**Why this matters for ML:** Transparent, conservative rules reduce accidental target leakage and make human review easier. Serialized JSON columns are not guessed into a model-ready form.

In [4]:
ROLE_BY_COLUMN = {
    "id": "metadata",
    "run_id": "grouping_time",
    "machine_id": "metadata",
    "timestamp": "grouping_time",
    "elapsed_seconds": "grouping_time",
    "phase": "constant_unusable",
    "stress_cpu_target_pct": "constant_unusable",
    "stress_memory_target_mb": "constant_unusable",
    "cpu_pct": "candidate_feature",
    "cpu_per_core_json": "review_needed",
    "cpu_frequency_mhz": "candidate_feature",
    "ram_pct": "candidate_feature",
    "ram_used_mb": "candidate_feature",
    "ram_available_mb": "candidate_feature",
    "swap_pct": "candidate_feature",
    "swap_used_mb": "candidate_feature",
    "disk_usage_pct": "candidate_feature",
    "disk_free_gb": "candidate_feature",
    "disk_read_mb_s": "candidate_feature",
    "disk_write_mb_s": "candidate_feature",
    "disk_latency_ms": "candidate_feature",
    "net_sent_mb_s": "candidate_feature",
    "net_recv_mb_s": "candidate_feature",
    "network_latency_ms": "candidate_feature",
    "process_count": "candidate_feature",
    "thread_count": "candidate_feature",
    "context_switches_per_s": "candidate_feature",
    "temperature_c": "candidate_feature",
    "gpu_usage_pct": "candidate_feature",
    "gpu_per_device_json": "review_needed",
    "battery_pct": "candidate_feature",
    "battery_plugged": "candidate_feature",
    "sensor_errors_json": "diagnostic",
    "missed_deadline": "target_related",
    "legacy_id": "constant_unusable",
    "sample_reliable": "diagnostic",
    "status": "leakage_risk",
    "ended_at_utc": "leakage_risk",
    "run_complete": "leakage_risk",
}

ROLE_REASON = {
    "candidate_feature": "Numeric system measurement available as an initial predictor candidate; validate missingness and stability before modeling.",
    "grouping_time": "Identifies observation order or run membership; retain for splitting/grouping, but do not use directly as an initial feature.",
    "diagnostic": "Quality or sensor-health indicator useful for auditing observations, not an initial predictive feature.",
    "metadata": "Identifier/context field retained for traceability; direct use could encourage memorization rather than generalization.",
    "leakage_risk": "May describe run completion or information known after/during the outcome; exclude until prediction timing is defined.",
    "constant_unusable": "Has at most one observed value (including an entirely missing column), so it provides no usable variation.",
    "target_related": "Potential outcome or target-proxy field; keep for later label design but never use as an initial predictor.",
    "review_needed": "Meaning or representation requires manual review; no feature-use decision is guessed.",
}

def example_values(series, limit=3):
    values = series.dropna().drop_duplicates().head(limit).tolist()
    return json.dumps([str(value)[:120] for value in values], ensure_ascii=False)

def group_availability(dataframe, value_column, group_column):
    availability = dataframe.groupby(group_column, dropna=False)[value_column].apply(lambda values: values.notna().any())
    unavailable = [str(group) for group, is_available in availability.items() if not is_available]
    return {
        f"available_{group_column}_count": int(availability.sum()),
        f"total_{group_column}_count": int(len(availability)),
        f"{group_column}_availability_percentage": round(float(availability.mean() * 100), 4) if len(availability) else np.nan,
        f"unavailable_{group_column}s": json.dumps(unavailable, ensure_ascii=False),
    }

unknown_rule_columns = sorted(set(metrics_df.columns) - set(ROLE_BY_COLUMN))
print("Columns not covered by a named rule (they will be review_needed):", unknown_rule_columns)

Columns not covered by a named rule (they will be review_needed): []


### 5. Calculate per-column quality and descriptive statistics

**Objective:** Build one audit record per source column.  
**What this checks:** Data type, missingness, presence, cardinality, constant status, non-finite numeric values, numeric range/center, and representative text values. Here `non_finite_count` includes numeric missing values plus positive/negative infinity; non-numeric columns receive `NA`.  
**Why this matters for ML:** These measurements expose unusable constants, missing sensors, invalid numeric values, suspicious ranges, and high-cardinality fields before they enter a pipeline.

In [5]:
row_count = len(metrics_df)
audit_records = []

for column_name in metrics_df.columns:
    series = metrics_df[column_name]
    is_numeric = pd.api.types.is_numeric_dtype(series)
    missing_count = int(series.isna().sum())
    unique_count = int(series.nunique(dropna=True))

    record = {
        "column": column_name,
        "dtype": str(series.dtype),
        "missing_count": missing_count,
        "missing_percentage": round(missing_count / row_count * 100, 4) if row_count else np.nan,
        "unique_count": unique_count,
        "present_percentage": round(series.notna().mean() * 100, 4) if row_count else np.nan,
        "is_constant": bool(unique_count <= 1),
        "non_finite_count": int((~np.isfinite(series.to_numpy(dtype=float))).sum()) if is_numeric else pd.NA,
        "minimum": float(series.min()) if is_numeric and series.notna().any() else np.nan,
        "maximum": float(series.max()) if is_numeric and series.notna().any() else np.nan,
        "mean": float(series.mean()) if is_numeric and series.notna().any() else np.nan,
        "median": float(series.median()) if is_numeric and series.notna().any() else np.nan,
        "example_values": pd.NA if is_numeric else example_values(series),
    }
    audit_records.append(record)

audit_df = pd.DataFrame(audit_records)
display(audit_df.head(10))

,column,dtype,missing_count,missing_percentage,unique_count,present_percentage,is_constant,non_finite_count,minimum,maximum,mean,median,example_values
0,id,int64,0,0.0,31059,100.0,False,0,1.000,31059.000,15530.000000,15530.000,NaN
1,run_id,str,0,0.0,10,100.0,False,<NA>,NaN,NaN,NaN,NaN,"[""6835f125-a038-4092-beff-5107ae998b39"", ""89cd..."
2,machine_id,str,0,0.0,3,100.0,False,<NA>,NaN,NaN,NaN,NaN,"[""0890dcc046c079acc4de4202"", ""7232bc533c21ce40..."
3,timestamp,str,0,0.0,31059,100.0,False,<NA>,NaN,NaN,NaN,NaN,"[""2026-07-25T14:45:59.803000Z"", ""2026-07-25T14..."
4,elapsed_seconds,float64,0,0.0,30997,100.0,False,0,0.096,17570.052,5803.469454,4960.061,NaN
5,phase,str,0,0.0,1,100.0,True,<NA>,NaN,NaN,NaN,NaN,"[""monitor""]"
6,stress_cpu_target_pct,float64,31059,100.0,0,0.0,True,31059,NaN,NaN,NaN,NaN,NaN
7,stress_memory_target_mb,float64,31059,100.0,0,0.0,True,31059,NaN,NaN,NaN,NaN,NaN
8,cpu_pct,float64,0,0.0,982,100.0,False,0,0.000,100.000,31.840462,17.800,NaN
9,cpu_per_core_json,str,0,0.0,29718,100.0,False,<NA>,NaN,NaN,NaN,NaN,"[""[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0..."


### 6. Check availability by machine and by run

**Objective:** Measure whether each column has at least one present value within every `machine_id` and every `run_id`.  
**What this checks:** It counts covered groups, calculates group coverage percentages, and records groups where a column is entirely absent.  
**Why this matters for ML:** A feature that exists only on some machines or runs can create biased training data, deployment failures, or spurious machine-specific patterns.

In [6]:
required_group_columns = {"machine_id", "run_id"}
missing_group_columns = required_group_columns - set(metrics_df.columns)
if missing_group_columns:
    raise KeyError(f"Cannot check group availability; missing columns: {sorted(missing_group_columns)}")

availability_records = []
for column_name in metrics_df.columns:
    coverage = {"column": column_name}
    coverage.update(group_availability(metrics_df, column_name, "machine_id"))
    coverage.update(group_availability(metrics_df, column_name, "run_id"))
    availability_records.append(coverage)

availability_df = pd.DataFrame(availability_records)
audit_df = audit_df.merge(availability_df, on="column", how="left", validate="one_to_one")
display(audit_df[[
    "column", "available_machine_id_count", "total_machine_id_count",
    "machine_id_availability_percentage", "available_run_id_count",
    "total_run_id_count", "run_id_availability_percentage"
]])

,column,available_machine_id_count,total_machine_id_count,machine_id_availability_percentage,available_run_id_count,total_run_id_count,run_id_availability_percentage
0,id,3,3,100.0000,10,10,100.0
1,run_id,3,3,100.0000,10,10,100.0
2,machine_id,3,3,100.0000,10,10,100.0
3,timestamp,3,3,100.0000,10,10,100.0
4,elapsed_seconds,3,3,100.0000,10,10,100.0
5,phase,3,3,100.0000,10,10,100.0
6,stress_cpu_target_pct,0,3,0.0000,0,10,0.0
7,stress_memory_target_mb,0,3,0.0000,0,10,0.0
8,cpu_pct,3,3,100.0000,10,10,100.0
9,cpu_per_core_json,3,3,100.0000,10,10,100.0


### 7. Assign roles and separate retention from feature use

**Objective:** Propose exactly one role and two separate decisions for every column.  
**What this checks:** Constants are forced to `constant_unusable`; known columns use the documented rules; all unfamiliar columns become `review_needed`. `keep_in_dataset` means retain for traceability/future work, while `use_as_model_feature_initially` is intentionally stricter.  
**Why this matters for ML:** Keeping useful context does not mean it is safe as a predictor. Separating these decisions protects grouping keys, diagnostics, and possible target/leakage fields from accidental feature inclusion.

In [7]:
def proposed_role(row):
    if row["is_constant"]:
        return "constant_unusable"
    return ROLE_BY_COLUMN.get(row["column"], "review_needed")

audit_df["proposed_role"] = audit_df.apply(proposed_role, axis=1)
audit_df["keep_in_dataset"] = ~audit_df["proposed_role"].eq("constant_unusable")
audit_df["use_as_model_feature_initially"] = audit_df["proposed_role"].eq("candidate_feature")
audit_df["reason"] = audit_df["proposed_role"].map(ROLE_REASON)

allowed_roles = set(ROLE_REASON)
assert len(audit_df) == metrics_df.shape[1], "Every source column must have exactly one audit row."
assert audit_df["column"].is_unique, "Audit contains duplicate column rows."
assert audit_df["proposed_role"].isin(allowed_roles).all(), "An unsupported role was assigned."
assert audit_df["reason"].notna().all(), "Every decision must have a reason."

display(audit_df[["column", "proposed_role", "keep_in_dataset", "use_as_model_feature_initially", "reason"]])

,column,proposed_role,keep_in_dataset,use_as_model_feature_initially,reason
0,id,metadata,True,False,Identifier/context field retained for traceabi...
1,run_id,grouping_time,True,False,Identifies observation order or run membership...
2,machine_id,metadata,True,False,Identifier/context field retained for traceabi...
3,timestamp,grouping_time,True,False,Identifies observation order or run membership...
4,elapsed_seconds,grouping_time,True,False,Identifies observation order or run membership...
5,phase,constant_unusable,False,False,Has at most one observed value (including an e...
6,stress_cpu_target_pct,constant_unusable,False,False,Has at most one observed value (including an e...
7,stress_memory_target_mb,constant_unusable,False,False,Has at most one observed value (including an e...
8,cpu_pct,candidate_feature,True,True,Numeric system measurement available as an ini...
9,cpu_per_core_json,review_needed,True,False,Meaning or representation requires manual revi...


### 8. Save the completed audit table

**Objective:** Export the final one-row-per-column audit to the required report path.  
**What this checks:** It writes only the derived audit and confirms the report exists with the expected number of rows.  
**Why this matters for ML:** A persistent audit makes feature decisions reviewable and gives downstream work a documented schema-quality checkpoint.

In [8]:
audit_df.to_csv(OUTPUT_PATH, index=False)

saved_audit = pd.read_csv(OUTPUT_PATH)
assert OUTPUT_PATH.is_file(), "Audit CSV was not created."
assert len(saved_audit) == metrics_df.shape[1], "Saved audit row count does not match the number of source columns."
print(f"Saved {len(saved_audit)} audited columns to: {OUTPUT_PATH}")

Saved 39 audited columns to: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/column_audit.csv


### 9. Review the proposed column groups

**Objective:** Display the key role lists requested for human review.  
**What this checks:** It surfaces candidate features, metadata/grouping fields, leakage risks, constants, and unresolved columns.  
**Why this matters for ML:** Compact review lists make it easier to challenge assumptions before feature engineering and prevent unsafe columns from quietly entering training.

In [9]:
def columns_with_roles(*roles):
    return audit_df.loc[audit_df["proposed_role"].isin(roles), "column"].tolist()

print("Candidate features:")
display(columns_with_roles("candidate_feature"))

print("Metadata/grouping columns:")
display(columns_with_roles("metadata", "grouping_time"))

print("Leakage-risk columns:")
display(columns_with_roles("leakage_risk"))

print("Constant columns:")
display(columns_with_roles("constant_unusable"))

print("Columns requiring manual review:")
display(columns_with_roles("review_needed"))

Candidate features:


['cpu_pct',
 'cpu_frequency_mhz',
 'ram_pct',
 'ram_used_mb',
 'ram_available_mb',
 'swap_pct',
 'swap_used_mb',
 'disk_usage_pct',
 'disk_free_gb',
 'disk_read_mb_s',
 'disk_write_mb_s',
 'disk_latency_ms',
 'net_sent_mb_s',
 'net_recv_mb_s',
 'network_latency_ms',
 'process_count',
 'thread_count',
 'context_switches_per_s',
 'temperature_c',
 'gpu_usage_pct',
 'battery_pct',
 'battery_plugged']

Metadata/grouping columns:


['id', 'run_id', 'machine_id', 'timestamp', 'elapsed_seconds']

Leakage-risk columns:


['status', 'ended_at_utc', 'run_complete']

Constant columns:


['phase', 'stress_cpu_target_pct', 'stress_memory_target_mb', 'legacy_id']

Columns requiring manual review:


['cpu_per_core_json', 'gpu_per_device_json']

### 10. Confirm outputs and source-file integrity

**Objective:** Finish with a reproducibility summary and a second source-file fingerprint.  
**What this checks:** It prints the exact input/output paths, audited-column count, and verifies byte-for-byte that the original CSV was not modified.  
**Why this matters for ML:** An integrity check proves that the audit was non-destructive and that later stages can rely on the same immutable input.

In [10]:
input_hash_after = sha256_file(INPUT_PATH)
source_unchanged = input_hash_before == input_hash_after

print(f"Input path: {INPUT_PATH}")
print(f"Output path: {OUTPUT_PATH}")
print(f"Number of audited columns: {len(audit_df)}")
print(f"Original dataset was not modified: {source_unchanged}")

assert source_unchanged, "The input CSV changed while the notebook was running."
assert OUTPUT_PATH.resolve() != INPUT_PATH.resolve(), "Output path must never overwrite the input dataset."

Input path: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/data/processed/cleaned_metrics.csv
Output path: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/column_audit.csv
Number of audited columns: 39
Original dataset was not modified: True
